# ROGII Wellbore Geology Prediction Notebook

In [ ]:
import os
import glob
import warnings
import math
from functools import lru_cache

import numpy as np
import pandas as pd

from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True

SEED = 42
rng = np.random.default_rng(SEED)

print("Imports loaded.")

In [ ]:
# ============================================================
# Configuration
# ============================================================

DATA_CANDIDATES = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
]

CV_WELL_LIMIT = 8          # calibration wells for data-driven ensemble weights
CV_PF_PARTICLES = 250      # lighter PF for CV calibration
CV_PF_SEEDS = 16
CV_SCALE = 5.0

SUB_PF_PARTICLES = 500     # stronger inference configuration
SUB_PF_SEEDS = 128
SUB_SCALE = 5.0

BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10,  8.0,  64.0, 2), ( 8, 35.0, 220.0, 1),
    (10, 14.0,  90.0, 5), (20,  4.0,  36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2), (20, 30.0, 200.0, 2), (15, 10.0,  80.0, 4),
    (25,  6.0,  50.0, 3), (10, 40.0, 300.0, 1), (12, 18.0, 120.0, 5),
    (30,  8.0,  70.0, 2), (10, 50.0, 400.0, 0),
]

METHOD_NAMES = ["pf", "beam", "knn", "offset"]
TRAIN_ONLY_METHODS = ["contact"]

print("Configuration set.")

In [ ]:
# ============================================================
# Data discovery and loaders
# ============================================================

def find_input_dir():
    for c in DATA_CANDIDATES:
        if os.path.isdir(c):
            print(f"INPUT_DIR = {c}")
            return c

    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    if hits:
        d = os.path.dirname(hits[0])
        print(f"Discovered INPUT_DIR = {d}")
        return d

    raise FileNotFoundError("Cannot locate competition data.")


INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, "train")
TEST_DIR = os.path.join(INPUT_DIR, "test")

_hw_tr = sorted(glob.glob(os.path.join(TRAIN_DIR, "*__horizontal_well.csv")))
_hw_te = sorted(glob.glob(os.path.join(TEST_DIR, "*__horizontal_well.csv")))

TRAIN_WELLS = [os.path.basename(f).split("__")[0] for f in _hw_tr]
TEST_WELLS = [os.path.basename(f).split("__")[0] for f in _hw_te]

print(f"Train wells: {len(TRAIN_WELLS)}")
print(f"Test wells : {len(TEST_WELLS)}")


def load_well(wid, split="train"):
    base = TRAIN_DIR if split == "train" else TEST_DIR
    hw_path = os.path.join(base, f"{wid}__horizontal_well.csv")
    hw = pd.read_csv(hw_path)

    tw_files = glob.glob(os.path.join(base, f"{wid}__typewell*.csv"))
    if len(tw_files) == 0:
        raise FileNotFoundError(f"No typewell file found for {wid} in {split}.")

    tw = pd.read_csv(tw_files[0])
    return hw, tw


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


print("Loaders ready.")

In [ ]:
import numpy as np
import pandas as pd

# ... (other imports remain unchanged)
from scipy.spatial.distance import cdist


def make_trajectory_features(df):
    """Diagnostic features (not all used directly in inference)."""
    out = df.copy()

    for col in ["X", "Y", "Z", "MD", "GR", "TVT_input"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out["dMD"] = out["MD"].diff().fillna(0.0)
    out["dX"] = out["X"].diff().fillna(0.0)
    out["dY"] = out["Y"].diff().fillna(0.0)
    out["dZ"] = out["Z"].diff().fillna(0.0)

    # Basic trajectory angles
    out["azimuth_step"] = np.degrees(np.arctan2(out["dY"], out["dX"])) % 360.0
    out["azimuth_step"] = out["azimuth_step"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    # Compute inclination angle (angle relative to the vertical Z axis)
    out["inclination_step"] = np.degrees(np.arctan2(np.sqrt(out["dX"]**2 + out["dY"]**2), out["dZ"]))
    out["inclination_step"] = out["inclination_step"].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    # More rolling statistics for GR
    out["GR_roll10"] = out["GR"].rolling(10, min_periods=1).mean()
    out["GR_roll25"] = out["GR"].rolling(25, min_periods=1).mean()
    out["GR_std10"] = out["GR"].rolling(10, min_periods=1).std().fillna(0.0)
    out["GR_std25"] = out["GR"].rolling(25, min_periods=1).std().fillna(0.0)
    out["GR_min10"] = out["GR"].rolling(10, min_periods=1).min()
    out["GR_max10"] = out["GR"].rolling(10, min_periods=1).max()
    out["GR_median10"] = out["GR"].rolling(10, min_periods=1).median()
    out["GR_diff"] = out["GR"].diff().fillna(0.0)
    out["GR_diff2"] = out["GR_diff"].diff().fillna(0.0)

    # Features related to TVT_input (pre-PS only)
    out["TVT_input_diff"] = out["TVT_input"].diff().fillna(0.0)
    out["TVT_input_roll10"] = out["TVT_input"].rolling(10, min_periods=1).mean()
    out["TVT_input_std10"] = out["TVT_input"].rolling(10, min_periods=1).std().fillna(0.0)

    # Turn rate (change in azimuth per MD step)
    out['azimuth_change'] = out['azimuth_step'].diff().fillna(0.0)
    out['turn_rate'] = out['azimuth_change'] / out['dMD'].replace(0, np.nan).fillna(0.0)
    out['turn_rate'] = out['turn_rate'].replace([np.inf, -np.inf], 0.0).fillna(0.0)

    return out

In [ ]:
# ============================================================
# Train-only physical model from geological contacts
# ============================================================

def tvt_from_contacts(hw_tr, tw_tr, ref_col=None, verbose=False):
    """
    Physical model: compute TVT from geological contact depths.

    If ref_col is None, auto-select the formation that minimises residual
    on the known (pre-PS) portion.
    """
    tw_g = tw_tr.dropna(subset=["Geology"]).copy()
    kn = hw_tr[hw_tr["TVT_input"].notna()].copy()

    if len(kn) == 0:
        raise ValueError("No known TVT_input rows found.")

    candidates = [
        col for col in tw_g["Geology"].dropna().unique()
        if col in hw_tr.columns
    ]
    if len(candidates) == 0:
        raise ValueError("No valid geology formation columns found.")

    def _fit_col(col):
        if col not in hw_tr.columns:
            return None, np.inf

        ref_rows = tw_g[tw_g["Geology"] == col]
        if len(ref_rows) == 0:
            return None, np.inf

        ref_tvt = ref_rows["TVT"].min()
        if np.isnan(ref_tvt):
            return None, np.inf

        pred = ref_tvt - (hw_tr["Z"] - hw_tr[col])
        offset = (kn["TVT_input"] - pred.loc[kn.index]).mean()
        pred = pred + offset

        rmse_known = np.sqrt(((kn["TVT_input"] - pred.loc[kn.index]) ** 2).mean())
        return pred, rmse_known

    if ref_col is not None:
        pred, _ = _fit_col(ref_col)
        if pred is not None:
            if verbose:
                print(f"Using ref_col={ref_col}")
            return pred

    best_pred = None
    best_rmse = np.inf
    best_col = None

    for col in candidates:
        pred, score = _fit_col(col)
        if score < best_rmse:
            best_pred = pred
            best_rmse = score
            best_col = col

    if best_pred is not None:
        if verbose:
            print(f"Selected formation: {best_col}, RMSE={best_rmse:.4f}")
        return best_pred

    raise ValueError("No valid geology formation column found.")


print("Contact model ready.")

In [ ]:
# ============================================================
# GR template matching: horizontal GR before PS vs known horizontal GR
# ============================================================

def gr_knn_tvt(hw, k=10, window=11, sigma_scale=1.0):
    """
    k-NN TVT prediction using pre-PS GR as a local correlation template.
    """
    kn = hw[hw["TVT_input"].notna()].copy()
    ev = hw[hw["TVT_input"].isna()].copy()

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy()

    gr_full = (
        hw["GR"]
        .interpolate(limit_direction="both")
        .fillna(hw["GR"].median())
        .values.astype(float)
    )

    kn_idx = list(kn.index)
    ev_idx = list(ev.index)
    n = len(gr_full)
    half = window

    def get_win(arr, i, half_window):
        start_idx = i - half_window
        end_idx = i + half_window + 1  # exclusive

        pad_left = max(0, -start_idx)
        pad_right = max(0, end_idx - n)

        slice_start = max(0, start_idx)
        slice_end = min(n, end_idx)

        window_slice = arr[slice_start:slice_end]
        return np.pad(window_slice, (pad_left, pad_right), mode="edge")

    kn_wins = np.stack([get_win(gr_full, i, half) for i in kn_idx])
    ev_wins = np.stack([get_win(gr_full, i, half) for i in ev_idx])

    def row_norm(X):
        mu = X.mean(axis=1, keepdims=True)
        std = X.std(axis=1, keepdims=True) + 1e-6
        return (X - mu) / std

    kn_wins_n = row_norm(kn_wins)
    ev_wins_n = row_norm(ev_wins)

    kn_tvt = kn["TVT_input"].values.astype(float)

    out = hw["TVT_input"].values.astype(float).copy()

    CHUNK = 2000  # bound the distance matrix to CHUNK x len(kn) floats
    for c0 in range(0, len(ev_idx), CHUNK):
        D = cdist(ev_wins_n[c0:c0 + CHUNK], kn_wins_n, metric="euclidean")
        for j in range(D.shape[0]):
            idx = ev_idx[c0 + j]
            nn = np.argsort(D[j])[:k]
            d = D[j][nn]
            sigma = max(sigma_scale * np.median(d), 1e-3)
            wts = np.exp(-0.5 * (d / sigma) ** 2)
            wts = wts / wts.sum()
            out[idx] = float(np.dot(wts, kn_tvt[nn]))

    return out


print("GR-kNN ready.")

In [ ]:
# ============================================================
# Particle Filter (horizontal GR + typewell GR correlation)
# ============================================================

def run_particle_filter(hw, tw, n_particles=500, seed=42):
    """Particle filter. Returns (predictions_array, total_log_likelihood)."""

    tw_s = tw.sort_values("TVT").copy()
    tw_tvt = tw_s["TVT"].values.astype(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).values.astype(float)

    kn = hw[hw["TVT_input"].notna()].copy()
    ev = hw[hw["TVT_input"].isna()].copy()

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy(), 0.0

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"])
    last_Z = float(last["Z"])
    last_MD = float(last["MD"])

    tw_at_k = np.interp(kn["TVT_input"].values, tw_tvt, tw_gr)
    gr_resid = kn["GR"].fillna(kn["GR"].median()).values.astype(float) - tw_at_k
    gs = float(np.clip(np.nanstd(gr_resid), 5.0, 80.0))
    if not np.isfinite(gs) or gs <= 0:
        gs = 20.0

    tail = kn.tail(30)
    if len(tail) >= 3:
        dt = np.diff(tail["TVT_input"].values.astype(float))
        dz = np.diff(tail["Z"].values.astype(float))
        dm = np.diff(tail["MD"].values.astype(float))
        valid = dm > 0
        ir = float(np.median((dt + dz)[valid] / dm[valid])) if valid.any() else 0.0
    else:
        ir = 0.0

    N = n_particles
    rng = np.random.default_rng(seed)

    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N, dtype=float) / N

    MOM = 0.998
    VN = 0.002
    PN = 0.005
    RP = 0.1
    RR = 0.001
    RESAMP = 0.5

    md_v = ev["MD"].values.astype(float)
    z_v = ev["Z"].values.astype(float)
    gr_v = ev["GR"].values.astype(float)

    out_vals = hw["TVT_input"].values.astype(float).copy()
    res = np.empty(len(ev), dtype=float)
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)

        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm_step + PN * rng.standard_normal(N)

        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos = tvt_p + z_v[i]

        if not np.isnan(gr_v[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr_v[i] - eg) / gs
            lk = np.exp(-0.5 * np.minimum(d ** 2, 600.0))
            lk = np.maximum(lk, 1e-300)

            avg_lk = float((w * lk).sum())
            log_lik += np.log(max(avg_lk, 1e-300))

            w = w * lk
            ws = w.sum()
            if ws > 0:
                w /= ws
            else:
                w = np.ones(N, dtype=float) / N

        n_eff = 1.0 / np.sum(w ** 2)
        if n_eff < RESAMP * N:
            cum = np.cumsum(w)
            u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(
                np.searchsorted(cum, u0 + np.arange(N) / N),
                0,
                N - 1
            )
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N, dtype=float) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik


def run_pf_ensemble(hw, tw, n_particles=500, n_seeds=128, scale=5.0):
    """Likelihood-weighted PF ensemble."""
    preds = []
    liks = []

    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks = np.array(liks, dtype=float)
    liks_n = liks - np.max(liks)

    weights = np.exp(liks_n / scale)
    weights /= weights.sum()

    return (weights[:, None] * np.stack(preds, axis=0)).sum(axis=0)


print("Particle filter ready.")

In [ ]:
# ============================================================
# Beam search over TVT track (GR matching path search)
# ============================================================

def beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs=10, mc=20.0, es=144.0, r=2):
    n = len(hgr)
    nt = len(tw_tvt)

    if n == 0:
        return np.array([last_tvt], dtype=float)

    if r > 0 and n > max(3, 2 * r + 1):
        win = min(2 * r + 1, n if n % 2 == 1 else n - 1)
        sgr = savgol_filter(hgr, win, min(2, win - 1))
    else:
        sgr = hgr.copy()

    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))

    MOVES = np.array([-2, -1, 0, 1, 2], dtype=np.int64)
    MC = mc * np.array([2.0, 1.0, 0.0, 1.0, 2.0])

    bidx = np.full(bs, si, dtype=np.int64)
    bcost = np.full(bs, np.inf, dtype=float)
    bcost[0] = 0.0
    bn = 1

    result = np.zeros(n, dtype=float)

    for step in range(n):
        gv = sgr[step]

        ni = bidx[:bn, None] + MOVES[None, :]
        ci = np.clip(ni, 0, nt - 1)
        valid = (ni >= 0) & (ni < nt)

        gr_e = (gv - tw_gr[ci]) ** 2 / es
        tot = bcost[:bn, None] + gr_e + MC[None, :]
        tot = np.where(valid, tot, np.inf)

        valid_flat = valid.flatten()
        ni_f = ni.flatten()[valid_flat]
        tot_f = tot.flatten()[valid_flat]

        if len(tot_f) == 0:
            result[step] = tw_tvt[bidx[0]]
            continue

        order = np.argsort(tot_f)
        ni_s = ni_f[order]
        tot_s = tot_f[order]

        _, first = np.unique(ni_s, return_index=True)
        ni_u = ni_s[first]
        tot_u = tot_s[first]

        kept = min(bs, len(ni_u))
        top = np.argsort(tot_u)[:kept]

        bidx[:kept] = ni_u[top]
        bcost[:kept] = tot_u[top]

        if kept < bs:
            bidx[kept:] = bidx[kept - 1]
            bcost[kept:] = np.inf

        bn = kept
        result[step] = tw_tvt[bidx[0]]

    return result


def run_beam_ensemble(hw, tw):
    """Average beam-search configs."""
    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]

    if len(ev) == 0:
        return hw["TVT_input"].values.astype(float).copy()

    last_tvt = float(kn.iloc[-1]["TVT_input"])

    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].values.astype(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).values.astype(float)

    gr_all = (
        hw["GR"]
        .interpolate(limit_direction="both")
        .fillna(tw_gr.mean())
        .values.astype(float)
    )
    hgr = gr_all[ev.index.to_numpy()]

    beam_results = [
        beam_search(hgr, tw_tvt, tw_gr, last_tvt, bs, mc, es, r)
        for (bs, mc, es, r) in BEAM_CONFIGS
    ]

    beam_mean = np.stack(beam_results, axis=0).mean(axis=0)

    out = hw["TVT_input"].values.astype(float).copy()
    out[ev.index.to_numpy()] = beam_mean
    return out


print("Beam search ready.")

In [ ]:
# ============================================================
# Offset-well prior (PPT: neighboring wells and geological dip)
# ============================================================

def compute_signature(hw):
    """Well location signature: XY centroid of the trajectory."""
    return np.array([hw["X"].mean(), hw["Y"].mean()], dtype=float)


_train_signatures = {}
for _wid in TRAIN_WELLS:
    try:
        _train_signatures[_wid] = compute_signature(load_well(_wid, "train")[0])
    except Exception:
        pass


def find_offset_wells_from_signature(sig, n_offset=5, exclude=None):
    """n nearest training wells by XY centroid distance (never the well itself)."""
    dists = [
        (wid, float(np.linalg.norm(sig - s)))
        for wid, s in _train_signatures.items()
        if wid != exclude and np.all(np.isfinite(s))
    ]
    dists.sort(key=lambda x: x[1])
    return [wid for wid, _ in dists[:n_offset]]


def offset_well_prior(hw_te, offset_wells):
    """Extrapolate from the last known TVT using nearby-well rates."""
    kn = hw_te[hw_te["TVT_input"].notna()].copy()
    ev = hw_te[hw_te["TVT_input"].isna()].copy()

    if len(ev) == 0 or len(kn) == 0 or not offset_wells:
        return None

    rates = []

    for wid in offset_wells:
        try:
            hw_off, _ = load_well(wid, "train")
            kn_off = hw_off[hw_off["TVT_input"].notna()].copy()

            if len(kn_off) < 10:
                continue

            tail = kn_off.tail(60)
            dtvt = np.diff(tail["TVT_input"].values.astype(float))
            dmd = np.diff(tail["MD"].values.astype(float))
            valid = dmd > 0

            if valid.sum() >= 5:
                rates.append(float(np.median(dtvt[valid] / dmd[valid])))

        except Exception:
            pass

    if not rates:
        return None

    med_rate = float(np.median(rates))

    last_tvt = float(kn.iloc[-1]["TVT_input"])
    last_md = float(kn.iloc[-1]["MD"])

    out = hw_te["TVT_input"].values.astype(float).copy()
    for idx in ev.index:
        md_delta = float(hw_te.loc[idx, "MD"] - last_md)
        out[idx] = last_tvt + med_rate * md_delta

    return out


print("Offset prior ready.")

In [ ]:
# ============================================================
# Diagnostics on a few training wells
# ============================================================

sample_wids = TRAIN_WELLS[:4]

fig, axes = plt.subplots(len(sample_wids), 2, figsize=(15, 4 * len(sample_wids)))
if len(sample_wids) == 1:
    axes = np.array([axes])

for i, wid in enumerate(sample_wids):
    hw, tw = load_well(wid, "train")
    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]

    ax1 = axes[i, 0]
    ax1.plot(kn["MD"], kn["TVT_input"], lw=1.5, label="TVT_input (known)")
    ax1.plot(hw["MD"], hw["TVT"], "--", lw=1, alpha=0.7, label="True TVT")
    if len(ev):
        ax1.axvline(ev["MD"].iloc[0], color="red", ls=":", lw=2, label="Prediction Start")
    ax1.set_title(f"{wid} : TVT vs MD")
    ax1.set_xlabel("MD")
    ax1.set_ylabel("TVT")
    ax1.legend()

    ax2 = axes[i, 1]
    ax2.plot(hw["MD"], hw["GR"], color="black", lw=0.8, alpha=0.8, label="Horizontal GR")
    ax2.plot(tw["TVT"], tw["GR"], color="red", lw=1, alpha=0.8, label="Typewell GR")
    if len(ev):
        ax2.axvline(ev["MD"].iloc[0], color="red", ls=":", lw=2, label="Prediction Start")
    ax2.set_title(f"{wid} : GR comparison")
    ax2.set_xlabel("MD / TVT")
    ax2.set_ylabel("GR")
    ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CV calibration on training wells (data-driven ensemble weights)
# ============================================================

cv_indices = np.linspace(
    0,
    len(TRAIN_WELLS) - 1,
    num=min(CV_WELL_LIMIT, len(TRAIN_WELLS)),
    dtype=int
)
CV_WELLS = [TRAIN_WELLS[i] for i in cv_indices]

print("Calibration wells:", CV_WELLS)

cv_results = {}
cv_preds = {}   # wid -> {"y_true": ..., method: hidden-zone prediction}

for wid in CV_WELLS:
    print(f"\nCalibrating on {wid} ...")
    hw, tw = load_well(wid, "train")

    ev = hw[hw["TVT_input"].isna()]
    if len(ev) == 0 or hw["TVT_input"].notna().sum() == 0:
        print("  skipped (no hidden zone or no known TVT)")
        continue

    y_true = hw.loc[ev.index, "TVT"].values.astype(float)

    preds = {}
    hidden = {"y_true": y_true}

    # Identity / null baseline: hold the last known TVT constant
    hidden["identity"] = np.full(len(ev), float(hw["TVT_input"].dropna().iloc[-1]))
    preds["identity"] = rmse(y_true, hidden["identity"])
    print(f"  identity: {preds['identity']:.4f}")

    try:
        contact_pred = tvt_from_contacts(hw, tw)
        hidden["contact"] = np.asarray(contact_pred[ev.index], dtype=float)
        preds["contact"] = rmse(y_true, hidden["contact"])
        print(f"  contact : {preds['contact']:.4f}")
    except Exception as e:
        print(f"  contact failed: {e}")

    try:
        pf_pred = run_pf_ensemble(
            hw, tw,
            n_particles=CV_PF_PARTICLES,
            n_seeds=CV_PF_SEEDS,
            scale=CV_SCALE
        )
        hidden["pf"] = np.asarray(pf_pred[ev.index], dtype=float)
        preds["pf"] = rmse(y_true, hidden["pf"])
        print(f"  pf      : {preds['pf']:.4f}")
    except Exception as e:
        print(f"  pf failed: {e}")

    try:
        beam_pred = run_beam_ensemble(hw, tw)
        hidden["beam"] = np.asarray(beam_pred[ev.index], dtype=float)
        preds["beam"] = rmse(y_true, hidden["beam"])
        print(f"  beam    : {preds['beam']:.4f}")
    except Exception as e:
        print(f"  beam failed: {e}")

    try:
        knn_pred = gr_knn_tvt(hw, k=10, window=11)
        hidden["knn"] = np.asarray(knn_pred[ev.index], dtype=float)
        preds["knn"] = rmse(y_true, hidden["knn"])
        print(f"  knn     : {preds['knn']:.4f}")
    except Exception as e:
        print(f"  knn failed: {e}")

    try:
        sig = compute_signature(hw)
        offs = find_offset_wells_from_signature(sig, n_offset=5, exclude=wid)
        offset_pred = offset_well_prior(hw, offs)
        if offset_pred is not None:
            hidden["offset"] = np.asarray(offset_pred[ev.index], dtype=float)
            preds["offset"] = rmse(y_true, hidden["offset"])
            print(f"  offset  : {preds['offset']:.4f}")
        else:
            print("  offset failed: no prior")
    except Exception as e:
        print(f"  offset failed: {e}")

    cv_results[wid] = preds
    cv_preds[wid] = hidden

print("\nCV calibration completed.")

In [ ]:
# ============================================================
# Data-driven weights from CV RMSE
# ============================================================

mean_rmse = {}
for method in METHOD_NAMES:
    vals = [
        cv_results[w][method]
        for w in cv_results
        if method in cv_results[w] and np.isfinite(cv_results[w][method])
    ]
    if len(vals) > 0:
        mean_rmse[method] = float(np.mean(vals))

print("=" * 50)
print("Mean RMSE per method")
print("=" * 50)

for method, score in sorted(mean_rmse.items(), key=lambda x: x[1]):
    print(f"{method:10s} : {score:.4f} ft")

T = 0.5
available = {m: v for m, v in mean_rmse.items() if np.isfinite(v)}

if available:
    methods_s = list(available.keys())
    rmse_arr = np.array([available[m] for m in methods_s], dtype=float)
    rmse_arr = rmse_arr - rmse_arr.min()
    weights = np.exp(-rmse_arr / T)
    weights /= weights.sum()
    ENSEMBLE_WEIGHTS = dict(zip(methods_s, weights))
else:
    ENSEMBLE_WEIGHTS = {m: 1.0 / len(METHOD_NAMES) for m in METHOD_NAMES}

print("\n" + "=" * 50)
print("CV-tuned Ensemble Weights")
print("=" * 50)
for m, w in sorted(ENSEMBLE_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f"{m:10s} : {w:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
methods = [m for m in ENSEMBLE_WEIGHTS.keys() if m in mean_rmse]
scores = [mean_rmse[m] for m in methods]
if len(scores) == len(methods) and len(scores) > 0:
    best = min(scores)
    colors = ["steelblue" if s == best else "salmon" for s in scores]
    bars = ax.bar(methods, scores, color=colors)
    ax.set_ylabel("Mean RMSE (ft)")
    ax.set_title("Cross-Validated RMSE by Method")
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width() / 2, score, f"{score:.3f}",
                ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough scores to plot.")

In [ ]:
# ============================================================
# Optional: inspect per-method predictions on a train well
# ============================================================

def plot_well_predictions(wid, split="train"):
    hw, tw = load_well(wid, split)
    ev = hw[hw["TVT_input"].isna()]

    if len(ev) == 0:
        print(f"{wid}: no prediction zone")
        return

    pf_pred = run_pf_ensemble(hw, tw, n_particles=CV_PF_PARTICLES, n_seeds=CV_PF_SEEDS, scale=CV_SCALE)
    beam_pred = run_beam_ensemble(hw, tw)
    knn_pred = gr_knn_tvt(hw, k=10, window=11)

    sig = compute_signature(hw)
    offs = find_offset_wells_from_signature(sig, n_offset=5, exclude=wid)
    offset_pred = offset_well_prior(hw, offs)

    ev_idx = ev.index.to_numpy()

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(hw["MD"], hw["TVT_input"], "b-", lw=1.5, label="TVT_input (known)")
    if "TVT" in hw.columns:
        ax.plot(hw.loc[ev_idx, "MD"], hw.loc[ev_idx, "TVT"], "k-", lw=2, label="True TVT")
    ax.plot(hw.loc[ev_idx, "MD"], pf_pred[ev_idx], "r--", lw=1.2, label="PF")
    ax.plot(hw.loc[ev_idx, "MD"], beam_pred[ev_idx], "g--", lw=1.2, label="Beam")
    ax.plot(hw.loc[ev_idx, "MD"], knn_pred[ev_idx], "m--", lw=1.2, label="GR k-NN")
    if offset_pred is not None:
        ax.plot(hw.loc[ev_idx, "MD"], offset_pred[ev_idx], color="orange", ls="--", lw=1.2, label="Offset prior")
    ax.axvline(ev["MD"].iloc[0], color="orange", ls=":", lw=2, label="PS")
    ax.set_title(f"{wid} — TVT Predictions")
    ax.set_xlabel("MD (ft)")
    ax.set_ylabel("TVT (ft)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


for wid in TRAIN_WELLS[:4]:
    plot_well_predictions(wid, "train")

In [ ]:
# ============================================================
# Submission generation
# ============================================================

sample = pd.read_csv(os.path.join(INPUT_DIR, "sample_submission.csv"))
TARGET_COL = [c for c in sample.columns if c != "id"][0]
sample["well"] = sample["id"].str.rsplit("_", n=1).str[0]
sample["row_idx"] = sample["id"].str.split("_").str[-1].astype(int)

train_wids = set(TRAIN_WELLS)
print(f"Training wells available: {len(train_wids)}")


def blend_hidden(preds_dict):
    """Weighted average of method predictions using CV-tuned weights."""
    total_w = 0.0
    total_p = None

    for m, arr in preds_dict.items():
        if arr is None:
            continue

        w = float(ENSEMBLE_WEIGHTS.get(m, 0.0))
        if w <= 0:
            continue

        arr = np.asarray(arr, dtype=float)
        total_p = arr * w if total_p is None else total_p + arr * w
        total_w += w

    if total_p is None:
        raise ValueError("All methods returned None")

    return total_p / total_w if total_w > 0 else total_p


rows = []

for wid in TEST_WELLS:
    print(f"\nProcessing {wid}...")
    hw_te, tw_te = load_well(wid, "test")

    tvt_phys = None
    hw_tr = None
    tw_tr = None

    if wid in train_wids:
        try:
            hw_tr, tw_tr = load_well(wid, "train")
            hw_te["TVT_input"] = hw_tr["TVT_input"].values
            tvt_phys = tvt_from_contacts(hw_tr, tw_tr)
            print("  Physical model OK (auto formation)")
        except Exception as e:
            print(f"  Physical model failed: {e}")

    tw_ref = tw_tr if tw_tr is not None else tw_te
    preds = {}

    try:
        preds["pf"] = run_pf_ensemble(
            hw_te, tw_ref,
            n_particles=SUB_PF_PARTICLES,
            n_seeds=SUB_PF_SEEDS,
            scale=SUB_SCALE
        )
        print("  PF ensemble OK")
    except Exception as e:
        print(f"  PF failed: {e}")
        lk = hw_te["TVT_input"].dropna()
        last_val = float(lk.iloc[-1]) if len(lk) else 0.0
        preds["pf"] = hw_te["TVT_input"].fillna(last_val).values.astype(float)

    try:
        preds["beam"] = run_beam_ensemble(hw_te, tw_ref)
        print("  Beam ensemble OK")
    except Exception as e:
        print(f"  Beam failed: {e}")
        preds["beam"] = preds["pf"].copy()

    try:
        preds["knn"] = gr_knn_tvt(hw_te, k=10, window=11)
        print("  GR k-NN OK")
    except Exception as e:
        print(f"  GR k-NN failed: {e}")
        preds["knn"] = None

    try:
        sig = compute_signature(hw_te)
        offs = find_offset_wells_from_signature(sig, n_offset=5, exclude=wid)
        prior = offset_well_prior(hw_te, offs)
        if prior is not None:
            preds["offset"] = prior
            print(f"  Offset prior OK (neighbors: {offs})")
        else:
            preds["offset"] = None
    except Exception as e:
        print(f"  Offset prior failed: {e}")
        preds["offset"] = None

    blended = blend_hidden({m: p for m, p in preds.items() if p is not None})

    ws = sample[sample["well"] == wid]
    for _, row in ws.iterrows():
        ridx = int(row["row_idx"])

        tvt_val = float(tvt_phys.iloc[ridx]) if tvt_phys is not None else np.nan
        if not np.isfinite(tvt_val):
            tvt_val = float(blended[ridx])

        rows.append({"id": row["id"], "tvt": tvt_val})

    print(f"  Added {len(ws)} rows for {wid}")

submission = pd.DataFrame(rows)
submission.rename(columns={"tvt": TARGET_COL}).to_csv("submission.csv", index=False)

print(f"\nDone: {len(submission)} rows saved to submission.csv")
print(submission.head())

In [ ]:
# ============================================================
# Submission quality checks
# ============================================================

print("=" * 60)
print("Submission Statistics")
print("=" * 60)

print(submission["tvt"].describe())

print("\nMissing values:")
print(submission.isna().sum())

print("\nDuplicate IDs:")
print(submission["id"].duplicated().sum())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(submission["tvt"], bins=60, edgecolor="black", alpha=0.8)
axes[0].set_title("Distribution of Submitted TVT Values")
axes[0].set_xlabel("TVT (ft)")
axes[0].set_ylabel("Count")

for wid in TEST_WELLS:
    sub_w = submission[submission["id"].str.startswith(wid)]
    axes[1].plot(np.arange(len(sub_w)), sub_w["tvt"].values, lw=1, label=wid)

axes[1].set_title("Predicted TVT Trajectory per Test Well")
axes[1].set_xlabel("Prediction Index")
axes[1].set_ylabel("TVT (ft)")
axes[1].legend(fontsize=7, loc="best")

plt.tight_layout()
plt.show()

print("\nSubmission shape:", submission.shape)
print("Expected rows:", len(sample))

assert len(submission) == len(sample), "Row count mismatch!"
assert submission["id"].nunique() == len(sample), "Duplicate IDs found!"
assert submission["tvt"].isna().sum() == 0, "NaNs found in submission!"

print("\n✓ Submission passed all checks")

In [ ]:
# ============================================================
# Cross-validated evaluation (training wells with known hidden-zone TVT)
# ============================================================

def _weights_from_rmse(mean_scores):
    """Same softmax-over-negative-RMSE rule as the weights cell (T)."""
    ms = [m for m in METHOD_NAMES if m in mean_scores and np.isfinite(mean_scores[m])]
    if not ms:
        return {m: 1.0 / len(METHOD_NAMES) for m in METHOD_NAMES}
    arr = np.array([mean_scores[m] for m in ms], dtype=float)
    w = np.exp(-(arr - arr.min()) / T)
    return dict(zip(ms, w / w.sum()))


def _blend_cv(hidden, weights):
    num, den = 0.0, 0.0
    for m, w in weights.items():
        if m in hidden and w > 0:
            num = num + w * hidden[m]
            den += w
    return num / den if den > 0 else None


per_well = {m: {} for m in ["identity", "contact"] + METHOD_NAMES}
per_well["ensemble (CV weights)"] = {}
per_well["ensemble (leave-one-well-out weights)"] = {}

for wid, hidden in cv_preds.items():
    y = hidden["y_true"]
    for m in ["identity", "contact"] + METHOD_NAMES:
        if m in hidden and np.isfinite(rmse(y, hidden[m])):
            per_well[m][wid] = rmse(y, hidden[m])

    # Weights tuned on all CV wells (what the submission uses; slightly optimistic)
    p = _blend_cv(hidden, ENSEMBLE_WEIGHTS)
    if p is not None:
        per_well["ensemble (CV weights)"][wid] = rmse(y, p)

    # Honest estimate: weights from the other CV wells only
    others = {}
    for m in METHOD_NAMES:
        v = [cv_results[w][m] for w in cv_results
             if w != wid and m in cv_results[w] and np.isfinite(cv_results[w][m])]
        if v:
            others[m] = float(np.mean(v))
    p = _blend_cv(hidden, _weights_from_rmse(others))
    if p is not None:
        per_well["ensemble (leave-one-well-out weights)"][wid] = rmse(y, p)

labels = {"identity": "identity (last known TVT)", "contact": "contact (train-only)"}
table = pd.DataFrame([
    {
        "method": labels.get(m, m),
        "mean_rmse_ft": float(np.mean(list(v.values()))) if v else np.nan,
        "std_rmse_ft": float(np.std(list(v.values()))) if v else np.nan,
        "n_wells": len(v),
    }
    for m, v in per_well.items()
])

print("=" * 66)
print(f"Cross-validated RMSE (ft) on {len(cv_preds)} training wells")
print("=" * 66)
print(table.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

table.to_csv("cv_results.csv", index=False)
print("\nSaved cv_results.csv")